### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Mock Example

In [3]:
prompt = """You are a helpful assistant.
Return an answer to the question
Question: what is your name?"""

In [4]:
response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

I’m ChatGPT.


In [5]:
response

ChatCompletion(id='chatcmpl-DuxxO49rvS3IYnCyN0EQJFh9TKMwE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m ChatGPT.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1782469214, model='gpt-5.4-nano-2026-03-17', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=8, prompt_tokens=26, total_tokens=34, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

### Add Instructor (Structured Outputs)

In [6]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [7]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question")

In [8]:
response = client.create(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [9]:
response

Answer(answer='I’m ChatGPT.')

In [10]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [12]:
response

Answer(answer='I’m ChatGPT.')

In [13]:
raw_response

Response(id='resp_0d595d1005cd4f74006a3e547584408191a87e7b6064c0fe5c', created_at=1782469749.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I’m ChatGPT."}', call_id='call_qJIKnxl8pG9GQ9DytwZEBlyd', name='Answer', type='function_call', id='fc_0d595d1005cd4f74006a3e54762c308191a9e17e6f3ae66793', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Correctly extracted `Answer` with all the required parameters with correct types')], top_p=0.98, background=False, comp

In [25]:
raw_response.usage

ResponseUsage(input_tokens=106, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=55, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=161)

In [14]:
class AnswerWithReasoning(BaseModel):
    reasoning: str = Field(description="Reasoning for the answer")
    answer: str = Field(description="Answer to the question")

In [15]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning
)

In [16]:
response

AnswerWithReasoning(reasoning="The user asks for my name. As an AI assistant, I don't have a personal name beyond being ChatGPT; respond succinctly.", answer='I’m ChatGPT.')

### RAG Pipeline

In [21]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [22]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [23]:
output = rag_pipeline("Any USB chargeable speaker?")

In [24]:
output

{'data_object': RAGGenerationResponse(answer='Yes—Raymate Bluetooth Speakers (B0C996WY16) is portable and rechargeable, with a built-in 3600mAh battery offering 1000+ minutes of playtime. It uses a Type-C charging port.'),
 'answer': 'Yes—Raymate Bluetooth Speakers (B0C996WY16) is portable and rechargeable, with a built-in 3600mAh battery offering 1000+ minutes of playtime. It uses a Type-C charging port.',
 'question': 'Any USB chargeable speaker?',
 'retrieved_context_ids': ['B0C996WY16',
  'B09X9838WY',
  'B0BRV544MV',
  'B0C9QZS95R',
  'B0CC4HBS85'],
 'retrieved_context': ['Raymate Bluetooth Speakers, HiFi Stereo Sound with DSP, 30W IPX7 Waterproof Speaker Wireless Bluetooth-V5.0, 1000mins Playtime, Portable Speaker for Home, Outdoor, Party 🎶HiFi Sound: With proven audio processing DSP chip technology, 30W dual speaker drivers and a more powerful amplifier module, the portable speakers delivers even, Balanced Sound Without Distortion. 🧱Integrated structure: With IPX7 Waterproof Spe